### Advanced Model ChatBundestag

Advanced model identifies all suitable markers for metadata retrieval in user query

In [30]:
import os
import json

# data handling & viz
import pandas as pd
import matplotlib.pyplot as plt

# language preprocessing
import re #regex
from wordcloud import WordCloud
import spacy # DE stopwords


# langchain packages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings

from langchain_groq import ChatGroq
from langchain_classic import hub


from pydantic import BaseModel, ValidationError
from typing import Optional, Literal, List


# environment variables
from dotenv import load_dotenv
import warnings

load_dotenv()
warnings.filterwarnings('ignore')

# -- LLM -- instantiate ChatGroq with llama
llm = ChatGroq(
    model= "llama-3.1-8b-instant",
    temperature=0,
    max_tokens=2048, # was None — explicit limit prevents silent truncation
    timeout=None,
    max_retries=2
)


# -- Data -- refs for vector db handling
vector_db_name = "debates_lp19"
vector_db_path = f"vector_databases/vector_db_{vector_db_name}"


In [2]:
# cleaned and reduced data set
df_exp_debates = pd.read_csv("data/debates_lp19.csv")

In [3]:
df_exp_debates.shape

(26869, 11)

### Chunking

In [4]:
# chunk size based on document size
TINY_DOC_THRESHOLD   = 50    # e.g. interjections
SMALL_DOC_THRESHOLD  = 400   # short statements = 1 chunk
MEDIUM_DOC_THRESHOLD = 1500  # medium speeches = paragraph-level split
# anything above: full recursive split with overlap

CHUNK_SIZE_MEDIUM = 500      # characters
CHUNK_SIZE_LARGE  = 400      # characters
CHUNK_OVERLAP     = 50       # roughly 10%

BATCH_SIZE = 500             # reduce to 250 if RAM issues


In [5]:
# split text
# todo: hard speech split to mark beginning and end of MPs speech (document)
def get_splitter(chunk_size):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", ";", ","], 
       
        
    )

paragraph_splitter = get_splitter(CHUNK_SIZE_MEDIUM)
token_splitter     = get_splitter(CHUNK_SIZE_LARGE)


In [6]:
# dynamic chunking (document-aware)
def chunk_single_document(doc: Document, doc_idx: int) -> list[Document]:
    """
    Applies dynamic chunking based on document length.
    Never merges across documents.

    Size buckets (characters):
        tiny   < 50     : 1 chunk, flagged (low semantic signal)
        small  < 400    : 1 chunk
        medium < 1500   : paragraph-level split
        large  ≥ 1500   : recursive token-level split with overlap
    """
    text = doc.page_content.strip()

    if not text:
        return []

    length = len(text)

    # Determine strategy
    if length < TINY_DOC_THRESHOLD:
        strategy = "tiny"
        splits = [doc]

    elif length < SMALL_DOC_THRESHOLD:
        strategy = "small"
        splits = [doc]

    elif length < MEDIUM_DOC_THRESHOLD:
        strategy = "medium"
        splits = paragraph_splitter.split_documents([doc])

    else:
        strategy = "large"
        splits = token_splitter.split_documents([doc])

    # attach metadata to every chunk
    total = len(splits)
    for i, chunk in enumerate(splits):
        chunk.metadata.update({
            "chunk_id":          f"doc{doc_idx}_chunk{i}",
            "chunk_index":       i,
            "total_chunks":      total,
            "is_full_document":  total == 1,
            "chunking_strategy": strategy,
            "doc_char_length":   length,
        })

    return splits


In [7]:
# batch pipeline
def df_to_document_batches(df, batch_size=BATCH_SIZE):
    """Generator: yields batches of Documents without loading full df into memory."""
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        yield [
            Document(
                page_content=row['text'],
                metadata={
                    'row_index':        i,
                    'speaker_name':     row['speech_identification_ent'],
                    'date':             row['date'],
                    'year': str(pd.to_datetime(row['date']).year), # new column with year from date column
                    'legislative_period': row['period'],
                    'session':          row['session'],
                    'role':             row['Role'],
                    'government_status': row['governing_Party'], # more precise
                    'party':            row['Party'],
                }
            )
            for i, row in batch.iterrows()
        ]

In [8]:
# run chunking pipeline
def run_chunking_pipeline(df, batch_size=BATCH_SIZE) -> list[Document]:
    """
    Full batched chunking pipeline.
    Tracks strategy distribution so you can inspect and tune thresholds.
    """
    all_chunks = []
    strategy_counts = {"tiny": 0, "small": 0, "medium": 0, "large": 0}
    total_docs = 0

    for batch_idx, doc_batch in enumerate(df_to_document_batches(df, batch_size)):
        for local_idx, doc in enumerate(doc_batch):
            global_idx = batch_idx * batch_size + local_idx
            chunks = chunk_single_document(doc, global_idx)
            all_chunks.extend(chunks)

            if chunks:
                strategy_counts[chunks[0].metadata["chunking_strategy"]] += 1

        total_docs += len(doc_batch)
        print(f"Batch {batch_idx + 1}: {total_docs} docs → {len(all_chunks)} chunks")

    print(f"\nChunking-Strategie: {strategy_counts}")
    print(f"Gesamt: {len(all_chunks)} Chunks aus {total_docs} Dokumenten")
    return all_chunks

In [9]:
# chunking
chunks = run_chunking_pipeline(df_exp_debates)

Batch 1: 500 docs → 6254 chunks
Batch 2: 1000 docs → 12357 chunks
Batch 3: 1500 docs → 18494 chunks
Batch 4: 2000 docs → 25388 chunks
Batch 5: 2500 docs → 33217 chunks
Batch 6: 3000 docs → 40820 chunks
Batch 7: 3500 docs → 46900 chunks
Batch 8: 4000 docs → 55160 chunks
Batch 9: 4500 docs → 63264 chunks
Batch 10: 5000 docs → 69676 chunks
Batch 11: 5500 docs → 76077 chunks
Batch 12: 6000 docs → 84460 chunks
Batch 13: 6500 docs → 90807 chunks
Batch 14: 7000 docs → 97241 chunks
Batch 15: 7500 docs → 103525 chunks
Batch 16: 8000 docs → 110116 chunks
Batch 17: 8500 docs → 115941 chunks
Batch 18: 9000 docs → 122754 chunks
Batch 19: 9500 docs → 128457 chunks
Batch 20: 10000 docs → 135038 chunks
Batch 21: 10500 docs → 141663 chunks
Batch 22: 11000 docs → 148356 chunks
Batch 23: 11500 docs → 156634 chunks
Batch 24: 12000 docs → 163146 chunks
Batch 25: 12500 docs → 169633 chunks
Batch 26: 13000 docs → 176123 chunks
Batch 27: 13500 docs → 184725 chunks
Batch 28: 14000 docs → 190722 chunks
Batch 29

In [10]:
# display results of chunking function
print(f"number of chunks created: {len(chunks)}")

number of chunks created: 356390


### Embedding

In [11]:
# instantiate embedding model 
embedding = HuggingFaceEmbeddings(
        model_name='intfloat/multilingual-e5-small', 
        # cloud upgrade path: intfloat/multilingual-e5-base or deepset/gbert-base or BAAI/bge-m3
        model_kwargs={'device': 'mps'}, 
        # cloud: 'mps' -> 'cuda'.
        encode_kwargs={"normalize_embeddings": True,
                      "prompt": "passage: "  # adds prefix automatically at indexing time
                      }
    )

In [12]:
# prefix handling for embedding model
class E5QueryWrapper(Embeddings):
    """
    Wraps HuggingFaceEmbeddings to add the required 'query: ' prefix
    for multilingual-e5 models at retrieval time only.
    """
    def __init__(self, base_embedding: Embeddings):
        self.base = base_embedding

    def embed_query(self, text: str) -> List[float]:
        return self.base.embed_query(f"query: {text}")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # prefix 'passage: ' already handled in encode_kwargs at indexing time
        return self.base.embed_documents(texts)

embedding_wrapped = E5QueryWrapper(embedding)

In [13]:
# function to create and save vector store 
def create_and_store(chunks,vector_db_path,embedding):
    """
    this function creates a vector store from chunks and saves it locally
    """
    # create the vector store 
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding, # parameter name is singular!
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    
     # save vector store locally
    vectorstore.save_local(vector_db_path)

    return vectorstore

In [14]:
# implement retrieval from FAISS db

def retrieve_from_vector_db(vector_db_path,embedding):
    """
    this function splits out a retriever object from a local vector store
    """
    
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding, # parameter name is plural!
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':15, # k nearest
        }  
    )
    
    return retriever,vectorstore


In [15]:
# check if vector store exists. if no: creates vector store
if not os.path.exists(vector_db_path):
        print("Vector DB not found. Creating and embedding chunks.")
        vectorstore=create_and_store(chunks=chunks, vector_db_path=vector_db_path, embedding=embedding_wrapped)
        print(f"Vector DB save to {vector_db_path}")
else:
    print(f"Vector DB found at {vector_db_path}. Skipping embedding.")

Vector DB not found. Creating and embedding chunks.
Vector DB save to vector_databases/vector_db_debates_lp19


In [16]:
# load the retriever and index
retriever,vectorstore = retrieve_from_vector_db(vector_db_path=vector_db_path, embedding=embedding_wrapped)

#type(retriever),type(vectorstore)

In [17]:
# check metadata in chunks
sample_docs = vectorstore.similarity_search("test", k=3)

for i, doc in enumerate(sample_docs):
    print(f"Chunk {i+1} metadata:")
    print(doc.metadata)
    print()

Chunk 1 metadata:
{'row_index': 26709, 'speaker_name': 'Frank Schäffler', 'date': '2021-06-25', 'year': '2021', 'legislative_period': 19, 'session': 237, 'role': 'MdB', 'government_status': 0, 'party': 'FDP', 'chunk_id': 'doc26709_chunk12', 'chunk_index': 12, 'total_chunks': 13, 'is_full_document': False, 'chunking_strategy': 'large', 'doc_char_length': 2964}

Chunk 2 metadata:
{'row_index': 14163, 'speaker_name': 'Hans-Jürgen Thies', 'date': '2019-12-19', 'year': '2019', 'legislative_period': 19, 'session': 137, 'role': 'MdB', 'government_status': 1, 'party': 'CDU/CSU', 'chunk_id': 'doc14163_chunk5', 'chunk_index': 5, 'total_chunks': 10, 'is_full_document': False, 'chunking_strategy': 'large', 'doc_char_length': 2497}

Chunk 3 metadata:
{'row_index': 6694, 'speaker_name': 'Roderich Kiesewetter', 'date': '2018-12-13', 'year': '2018', 'legislative_period': 19, 'session': 71, 'role': 'MdB', 'government_status': 1, 'party': 'CDU/CSU', 'chunk_id': 'doc6694_chunk8', 'chunk_index': 8, 'total

### Prompt

In [18]:
# maps natural-language user terms → (metadata_key, metadata_value)
# extend here when adding new filter terms - no changes needed elsewhere

FILTER_MAP = {
    # parties
    "spd":       ("Party", "SPD"),
    "cdu":       ("Party", "CDU"),
    "csu":       ("Party", "CSU"),
    "grüne":     ("Party", "Grüne"),
    "fdp":       ("Party", "FDP"),
    "linke":     ("Party", "Linke"),
    "afd":       ("Party", "AfD"),
    "fraktionslos": ("Party", "fraktionslos"),
    "parteilos": ("Party", "parteilos"),
    # -- add other parties to complete list since 1949
    
    # government status
    "kabinett":       ("Party", "Cabinet"),   # governing_Party == Cabinet
    "regierungspartei": ("governing_Party", 1),
    "opposition":       ("governing_Party", 0),
    
    # roles
    "kanzler":         ("Role", "Bundeskanzler"),
    "bundeskanzler":   ("Role", "Bundeskanzler"),
    "kanzlerin":       ("Role", "Bundeskanzler"),
    "bundeskanzlerin": ("Role", "Bundeskanzler"),
    "minister":        ("Role", "Bundesminister"),
    "bundesminister":  ("Role", "Bundesminister"),
    "staatssekretär":  ("Role", "Staatssekretär"),
    "staatsminister":  ("Role", "Staatsminister"),
    "abgeordnete":     ("Role", "MdB"),
    "mdb":             ("Role", "MdB"),
    "mitglied des bundestags":("Role", "MdB"),
    
    # legislative periods
    "19. wahlperiode": ("period", "19"),
    "19 wahlperiode":  ("period", "19"),
    "wp19":            ("period", "19"),
    "18. wahlperiode": ("period", "18"),
    "18 wahlperiode":  ("period", "18"),
    "wp18":            ("period", "18"),
    
    # time frame 
    "2021": ("year", "2021"),
    "2020": ("year", "2020"),
    "2019": ("year", "2019"),
    "2018": ("year", "2018"),
    "2017": ("year", "2017"),
    # -- add remaining time frames til 1949 for full data set
}

In [61]:
# extract metadata filters and semantic query from free-text user input 
# handles: party, role, government_status, cabinet, speaker, session, date, legislative_period
# returns (semantic_query, filters) tuple
def parse_query_filters(user_input: str) -> tuple[str, dict]:
    """
    Parses a free-text user query into a semantic search string and a
    metadata filter dict for FAISS.

    Extraction order (all additive, multiple filters supported):
      1. Party affiliation     — matched via FILTER_MAP
      2. Government status     — Regierungspartei / Opposition
      3. Role                  — Minister, Staatssekretär, Abgeordnete
      4. Legislative period    — "19. Wahlperiode", "WP19", etc.
      5. Session               — "Sitzung 42", "42. Sitzung"
      6. Date                  — ISO (2019-03-14) or German (14.03.2019)
      7. Speaker name          — "von [Name]" or "Redner [Name]"

    Returns:
        semantic_query : str   — residual text after filter terms removed
        filters        : dict  — metadata key/value pairs for FAISS filter
    """
    filters = {}
    semantic = user_input.lower()

    # 1–4: FILTER_MAP lookup
    for term, (key, value) in FILTER_MAP.items():
        if term in semantic:
            filters[key] = value
            semantic = semantic.replace(term, "")

    # 5: Session — "Sitzung 42" or "42. Sitzung"
    session_match = re.search(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', semantic)
    if session_match:
        session_num = session_match.group(1) or session_match.group(2)
        filters["session"] = session_num
        semantic = re.sub(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', '', semantic)

    # 6: Date — ISO (2019-03-14) or German (14.03.2019)
    date_match = re.search(r'\d{4}-\d{2}-\d{2}|\d{2}\.\d{2}\.\d{4}', semantic)
    if date_match:
        filters["date"] = date_match.group()
        semantic = semantic.replace(date_match.group(), "")

    # 7: Speaker — capture full name (Vorname + Nachname) anywhere in query
    speaker_match = re.search(
        r'\b([A-ZÄÖÜ][a-zäöüß\-]+(?:\s[A-ZÄÖÜ][a-zäöüß\-]+)+)\b',
        user_input
    )
    if speaker_match:
        filters["speaker_name"] = speaker_match.group(1)
        semantic = re.sub(
            re.escape(speaker_match.group(1)), '', semantic, flags=re.IGNORECASE
        )

    semantic = re.sub(r'\s+', ' ', semantic).strip()
    return semantic, filters

In [62]:
# inject chunk metadata into context string (-> LLM)
# attributes extracted arguments to specific speakers/ sessions
def format_context_with_metadata(docs: list[Document]) -> str:
    """
    Prepends a metadata header to each retrieved chunk so the LLM can
    attribute extracted arguments to specific speakers, parties and sessions.
    Fields with missing/NaN values are omitted from the header.
    """
    formatted = []
    for doc in docs:
        m = doc.metadata
        header_parts = []
        # headers as defined in df_to_documents_batch()
        if m.get("speaker_name"):       header_parts.append(f"Redner: {m['speaker_name']}")
        if m.get("party"):              header_parts.append(f"Partei: {m['party']}")
        if m.get("government_status"):   header_parts.append(f"Regierungspartei: {m['government_status']}")
        if m.get("role"):               header_parts.append(f"Rolle: {m['role']}")
        if m.get("date"):               header_parts.append(f"Datum: {m['date']}")
        if m.get("year"):               header_parts.append(f"Jahr: {m['year']}")
        if m.get("session"):            header_parts.append(f"Sitzung: {m['session']}")
        if m.get("legislative_period"): header_parts.append(f"Wahlperiode: {m['legislative_period']}")
        
        header = f"[{' | '.join(header_parts)}]" if header_parts else ""
        formatted.append(f"{header}\n{doc.page_content}".strip())
        
    return "\n\n---\n\n".join(formatted)

In [63]:
# retriever with dynamic metadata filters. Falls back to unfiltered retriever if no filters detected
def get_filtered_retriever(vectorstore, filters: dict, k: int = 15):
    """
    Returns a retriever with metadata filters applied.
    Falls back to unfiltered retrieval if filters dict is empty.
    """
    search_kwargs = {"k": k}
    if filters:
        search_kwargs["filter"] = filters
    return vectorstore.as_retriever(search_kwargs=search_kwargs)

In [64]:
# prompt
prompt = ChatPromptTemplate.from_template(
    """
    Du bist ein Assistent für Argumentationsanalyse, spezialisiert auf parlamentarische Debatten des Deutschen Bundestags.

    Deine Aufgabe ist es, die Argumentationsstruktur aus dem untenstehenden Redeauszug zu extrahieren,
    wobei du die Frage des Nutzers/ der Nutzerin als thematischen Fokus verwendest.
    
    Du erhältst einen Redeausschnitt aus einem Bundestagsprotokoll (Kontext) 
    und eine Frage zur Position des Sprechers, einer Partei oder der Regierung / Opposition. Extrahiere das zentrale Argument 
    im vorgegebenen JSON-Format. Erfinde nichts; halte dich ausschließlich 
    an den gegebenen Kontext.

    Definitionen:
    - claim: die zentrale politische Position oder der Vorschlag, für den oder gegen den argumentiert wird
    - grounds: faktische oder normative Belege, die den claim stützen
    - rebuttal: antizipierter Einwand der Gegenseite + Widerlegung durch den Sprecher
    - attack: offensive Kritik des Sprechers an der Gegenposition 

    ### Beispiel 1:
    Frage 1: "Wie begründet Heiko Maas die Notwendigkeit, die bestehende Rüstungskontrollarchitektur zu erhalten?"

    Kontext 1: [Redner: Heiko Maas | Partei: SPD | Rolle: Bundesminister des Auswärtigen |
    Datum: 2018-04-19 | Sitzung: 26 | Wahlperiode: 19 | Regierungsstatus: Regierungspartei]

    Russland hat die konventionelle Rüstungskontrolle in Europa seit über zehn Jahren
    einseitig suspendiert. Die USA und Russland werfen sich gegenseitig vor, den INF-Vertrag
    zu verletzen, anstatt die Vorwürfe konstruktiv auszuräumen. Gleichzeitig rüsten China,
    Indien und Pakistan erheblich auf, und die USA setzen die Modernisierung ihres
    Nuklearwaffenarsenals fort. Kaum eines der Regime, die jahrzehntelang zu unserer
    Sicherheit in Europa beigetragen haben, funktioniert noch in vollem Umfang. Maßgeblich
    auf deutsches Betreiben hin sind die Verhandlungen über ein Verbot zur Herstellung von
    waffenfähigem Spaltmaterial ein gutes Stück nähergerückt. Auch die USA und Russland
    haben im Februar 2018 die im Rahmen des New-START-Vertrages vereinbarten
    Obergrenzen für einsatzbereite strategische Nuklearwaffen erreicht – ein gutes Zeichen
    für die anstehende Verlängerung des Vertrages.

    Output 1:
    {{
    "claim": "Die bestehende Rüstungskontrollarchitektur muss aktiv erhalten und modernisiert werden, da ihre Erosion die Sicherheit in Europa und weltweit direkt gefährdet.",
    "grounds": [
    "Russland hat die konventionelle Rüstungskontrolle in Europa seit über zehn Jahren einseitig suspendiert; USA und Russland beschuldigen sich gegenseitig, den INF-Vertrag
    zu verletzen.",
    "China, Indien, Pakistan und die USA rüsten erheblich auf; kaum eines der jahrzehntelangen Sicherheitsregime in Europa funktioniert noch vollständig.",
    "Auf deutsches Betreiben hin sind Verhandlungen über ein Verbot waffenfähigen Spaltmaterials vorangekommen; USA und Russland haben New-START-Obergrenzen erreicht – ein positives Signal für die Vertragsverlängerung."
    ],
    "rebuttal": [
    "Obwohl die sicherheitspolitische Lage schwierig ist, zeigen jüngste Fortschritte bei New START, dass Abrüstungsschritte auch im aktuellen Klima möglich sind. Daher ist das Festhalten an der Architektur keine naive Hoffnung, sondern realpolitisch geboten."
    ],
    "attack": [
    "Wer die regelbasierte Abrüstungsarchitektur aufgibt oder aushöhlt, überlässt das
    Feld denjenigen, die auf Aufrüstung setzen – und riskiert, dass eingehegteKonflikte
    wie der iranische Nuklearstreit wieder aufbrechen."
    ],
    "speaker": "Heiko Maas",
    "party": "SPD",
    "government_status": "Regierungspartei",
    "role": "Bundesminister des Auswärtigen",
    "date": "2018-04-19",
    "session": "26",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}
    
    ### Beispiel 2:
    Frage 2: "Wie kritisiert Armin-Paulus Hampel die außenpolitischen Ambitionen der Bundesregierung?"

    Kontext 2: [Redner: Armin-Paulus Hampel | Partei: AfD | Rolle: Bundestagsabgeordneter |
    Datum: 2018-04-19 | Sitzung: 26 | Wahlperiode: 19 | Regierungsstatus: Opposition]

    Ich staune immer, wenn die Deutschen ihre Weltsicht darlegen. Wir wollen in der
    Korea-Frage mitmischen, wir wollen in Syrien dabei sein, wir wollen zu einem Ausgleich
    in der Welt beitragen – bei Dingen, bei denen Deutschland im Grunde genommen kein
    Gewicht hat. Wenn die Amerikaner sich entscheiden sollten, den Vertrag mit dem Iran
    aufzugeben, dann werden wir Deutschen daran nichts ändern. Auch an dem Dialog
    zwischen Nord- und Südkorea oder gar den USA und Nordkorea werden wir
    wahrscheinlich nicht teilhaben.

    Output 2:
    {{
    "claim": "Deutschlands außenpolitische Ambitionen sind unrealistisch, weil das Land in den entscheidenden globalen Konflikten schlicht nicht das politische Gewicht besitzt, um tatsächlich Einfluss zu nehmen.",
    "grounds": [
    "In der Korea-Frage, in Syrien und beim Iran-Deal agiert Deutschland ohne reale Einflussmöglichkeiten: Sollten die USA den Iran-Vertrag aufgeben, kann Deutschland das nicht verhindern.",
    "An direkten Verhandlungen zwischen den USA und Nordkorea wird Deutschland voraussichtlich nicht beteiligt sein."
    ],
    "rebuttal": [
    "Die Bundesregierung betont deutschen Einfluss auf EU-Ebene und in multilateralen Foren. Hampel entgegnet implizit, dass dieser institutionelle Rahmen die fehlende bilaterale Machtstellung nicht kompensiert."
    ],
    "attack": [
    "Die Außenpolitik der Bundesregierung leidet an einer Diskrepanz zwischen rhetorischem Gestaltungsanspruch und tatsächlichem politischem Gewicht – ein Missverhältnis, das in der Debatte nicht offen benannt wird."
    ],
    "speaker": "Armin-Paulus Hampel",
    "party": "AfD",
    "government_status": "Opposition",
    "role": "Bundestagsabgeordneter",
    "date": "2018-04-19",
    "session": "26",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}
    
    ### Beispiel 3:
    Frage 3: "Welche Position vertritt die Bundesregierung zum Atomwaffenverbotsvertrag?"

    Kontext 3: [Redner: Heiko Maas | Partei: SPD | Rolle: Bundesminister des Auswärtigen |
    Datum: 2018-04-19 | Sitzung: 26 | Wahlperiode: 19 | Regierungsstatus: Regierungspartei]

    Wir gehen nicht den Weg des Atomwaffenverbotsvertrages. Unser zentraler Eckpfeiler
    und Kompass ist und bleibt der nukleare Nichtverbreitungsvertrag. Da gilt es, das
    Erreichte – das ist nicht wenig – zu bewahren. Wir haben hier in Deutschland den
    Anspruch, die Rahmenbedingungen für die nuklearwaffenfreie Welt aktiv mitzugestalten.
    Das Ziel einer nuklearwaffenfreien Welt dürfen wir nicht aus den Augen verlieren.

    Output 3:
    {{
    "claim": "Die Bundesregierung lehnt den Atomwaffenverbotsvertrag ab und setzt stattdessen auf den Nichtverbreitungsvertrag als verbindlichen Rahmen für nukleare Abrüstung.",
    "grounds": [
    "Der nukleare Nichtverbreitungsvertrag ist der zentrale Kompass der deutschen Abrüstungspolitik; das bisher Erreichte soll bewahrt werden.",
    "Deutschland beansprucht eine aktive Rolle bei der Gestaltung der Rahmenbedingungen für eine nuklearwaffenfreie Welt – auch gegenüber den G7-Partnern."
    ],
    "rebuttal": [
    "Das Ziel einer nuklearwaffenfreien Welt wird ausdrücklich nicht aufgegeben. Die Ablehnung des Verbotsvertrages bedeutet also keine Abkehr vom Abrüstungsziel, sondern eine andere Wegwahl: schrittweise über bestehende Verträge statt über ein sofortiges Verbot."
    ],
    "attack": [
    "Implizit richtet sich die Position gegen Ansätze, die den Verbotsvertrag als schnelleren oder wirksameren Weg zur nuklearen Abrüstung betrachten."
    ],
    "speaker": "Heiko Maas",
    "party": "SPD",
    "government_status": "Regierungspartei",
    "role": "Bundesminister des Auswärtigen",
    "date": "2018-04-19",
    "session": "26",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}
        
    Frage:
    {question}
        
    Regeln:
    - Verwende NUR Informationen aus dem Kontext, keine externen Kenntnisse
    - Erfinde NIEMALS Metadaten. Wenn ein Feld nicht im Kontext-Header steht, setze es auf null.
    - Zitiere oder paraphrasiere grounds, rebuttal und attack eng aus dem Text, nichts erfinden
    - Claim darf als knappe Synthese formuliert werden
    - Fokussiere die Extraktion auf die Frage des Nutzers, ignoriere thematisch irrelevante Textteile
    - Entnimm IMMER Redner/in, Partei, Rolle, Datum, Sitzung, Regierungspartei/ Opposition und Wahlperiode aus den Metadaten-Kopfzeilen im Kontext
    - Setze Felder auf null, wenn die entsprechenden Informationen im Kontext nicht vorhanden sind
    - Wenn kein klarer claim erkennbar ist, setze "claim" auf null
    - Wenn keine grounds vorhanden sind, gib eine leere Liste zurück
    - Wenn keine rebuttals oder attacks vorhanden sind, gib eine leere Liste zurück
    - Gib NUR valides JSON zurück, keine Erklärung, keine Einleitung, keine Markdown-Formatierung
    - Wenn der Kontext keine relevanten Informationen zur Anfrage enthält, gib folgendes zurück:
        {{"claim": null, "grounds": [], "rebuttal": [], "attack": [], 
        "confidence": "low", "note": "Keine relevanten Informationen gefunden. 
        Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren."}}

    WICHTIG: Deine Antwort muss ausschließlich mit {{ beginnen. Kein Text davor.
    
    Kontext:
    {context}
    
    """)

In [65]:
# pydantic schema for output
# core argument based on toulmin's model of argumentation
class ArgumentStructure(BaseModel):
    # core argument - all optional to account for null findings
    claim:               Optional[str] = []
    grounds:             Optional[list[str]] = []
    rebuttal:            Optional[list[str]] = []   # conditions under which claim does not hold
    attack:              Optional[list[str]] = []
    # speaker attribution
    speaker:             Optional[str]
    party:               Optional[str]
    government_status:   Optional[str]
    role:                Optional[str]
    date:                Optional[str]
    session:             Optional[str]
    legislative_period:  Optional[str]
    # quality
    confidence:          Literal["high", "medium", "low"] 
    reasoning:           Optional[str] = None
    note:                Optional[str] = None # carries message if no result
    



In [66]:
# check and format LLM output: according to Pydantic schema
def parse_llm_output(raw_output: str) -> dict:
    """
    Cleans, parses and validates LLM output against ArgumentStructure schema.
    Returns {"status": "ok", "data": ...} or error dict with raw output for debugging.
    """
    try:
        # strip markdown fences
        cleaned = re.sub(r"```json|```", "", raw_output).strip()
        
        # extract JSON block even if model adds prose before/after
        json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not json_match:
            return {"status": "json_error", "raw": raw_output, "error": "No JSON block found in output"}
        
        json_str = json_match.group()
        parsed = json.loads(json_str)
        validated = ArgumentStructure(**parsed)
        return {"status": "ok", "data": validated.model_dump()}

    except json.JSONDecodeError as e:
        return {"status": "json_error", "raw": raw_output, "error": str(e)}
    except ValidationError as e:
        return {"status": "validation_error", "raw": raw_output, "error": str(e)}


In [67]:
# runnable RAG pipeline (LCEL)
# parse user input -> retrieve
def connect_chains(retriever, vectorstore):
    """
    Full RAG chain with dynamic metadata filtering.
    Connects retriever, prompt and llm into a RAG chain for argument extraction.
    User question both drives semantic search and focuses the extraction.

    Flow:
      1. Parse user input → (semantic_query, filters)
      2. Build filtered retriever from vectorstore
      3. Retrieve top-k chunks matching semantic query + metadata filters
      4. Format chunks with metadata headers
      5. Pass formatted context + original question to prompt
      6. LLM generates structured argument JSON
      7. Parse and validate output against ArgumentStructure schema
    """
    def retrieve_and_format(user_input: str) -> dict:
        semantic_query, filters = parse_query_filters(user_input) #extract metadata filters and semantic query from user input
        filtered_retriever = get_filtered_retriever(vectorstore, filters) #retriever applying meta data filters
        docs = filtered_retriever.invoke(semantic_query)
        return {
            "context":  format_context_with_metadata(docs),
            "question": user_input   # original question — preserves full user intent for LLM
        }

    rag_chain = (
        RunnableLambda(retrieve_and_format)
        | prompt
        | llm
        | RunnableLambda(lambda msg: parse_llm_output(msg.content))
    )
    return rag_chain

In [68]:
# pass vectorstore as second argument
rag_chain = connect_chains(retriever, vectorstore)

In [69]:
# chat loop
ATTRIBUTION_FIELDS = [
    ("speaker",            "👤 Redner"),
    ("party",              "🏛️ Partei"),
    ("role",               "💼 Rolle"),
    ("government_status",   "📍 Kabinett/ Opposition"),
    ("date",               "📅 Datum"),
    ("session",            "📋 Sitzung"),
    ("legislative_period", "🗳️ Wahlperiode"),
]


In [70]:
# rag chain            
def chat_with_rag(chain):
    """
    Interactive RAG chat loop with argument mining output.
    Handles: structured output, no-results case, JSON errors.
    """
    print("Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.")
    print("Schreibe 'exit', um den Chat zu beenden.\n")

    while True:
        user_input = input("Du: ").strip()

        if not user_input:
            continue # skip accidental empty enters to prevent crash

        if user_input.lower() in ["exit", "quit"]:
            print("Chat wird beendet. Auf Wiedersehen!")
            break

        try:
            result = chain.invoke(user_input)

            # ── JSON / validation error ──────────────────────────────────────
            if result["status"] != "ok":
                print(f"\n⚠️  Parsing fehlgeschlagen ({result['status']})")
                print(f"Fehler: {result['error']}")
                print(f"Rohausgabe:\n{result['raw']}\n") 
                continue

            data = result["data"]

            # ── No relevant results ──────────────────────────────────────────
            if data.get("note"):          
                print(f"\n⚠️  {data['note']}\n") 
                continue

            if not data["claim"]:
                print("\n⚠️  Keine relevanten Informationen gefunden.")
                print("Tipp: Versuche die Anfrage zu vereinfachen oder weniger Filter zu verwenden.\n")
                continue

            # ── Normal structured output ─────────────────────────────────────
            print("\n" + "─" * 60)

            # Speaker / context info
            speaker   = data.get("speaker")   or "unbekannt"
            party     = data.get("party")      or "?"
            government_status = data.get("government_status") or "?"
            role      = data.get("role")       or "?"
            date      = data.get("date")       or "?"
            period    = data.get("legislative_period") or "?"
            session   = data.get("session") or "?"
            confidence = data.get("confidence") or "?"
            
            print(f"👤  {speaker} ({party} · {role} · {government_status})")
            print(f"📅  {date}  |  Legislaturperiode: {period} | Sitzung: {session} ")
            print(f"🎯  Konfidenz: {confidence}")
            print("─" * 60)

            # Claim
            print(f"\n📌  Claim:\n    {data['claim']}\n")

            all_arguments = data.get("grounds" or []) + data.get("rebuttal" or []) + data.get("attack" or []) #can be empty if no findings
            if all_arguments:
                print("💬  Argumente:")
                for a in all_arguments:
                    print(f"    • {a}")
            else:
                print("💬  Argumente: keine gefunden")

            print("─" * 60 + "\n")

        except Exception as e:
            print(f"\n❌  Fehler: {e}\n")
            


In [ ]:
# run chat
chat_with_rag(rag_chain)

Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.
Schreibe 'exit', um den Chat zu beenden.



Du:  Welche Position vertritt Gregor Gysi zum Atomwaffenverbotsvertrag?



⚠️  Keine relevanten Informationen gefunden. Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren.



Du:  Welche Position vertritt Gregor Gysi ?



────────────────────────────────────────────────────────────
👤  Gregor Gysi (DIE LINKE · Bundestagsabgeordneter · Opposition)
📅  2018-04-19  |  Legislaturperiode: 19 | Sitzung: 26 
🎯  Konfidenz: high
────────────────────────────────────────────────────────────

📌  Claim:
    Die Bundesregierung sollte sich für eine stärkere Einbindung in die europäische Verteidigungspolitik entscheiden, um die Sicherheit in Europa zu stärken.

💬  Argumente:
    • Die Europäische Union sollte eine stärkere Rolle in der internationalen Sicherheitspolitik spielen, um die Interessen der Mitgliedstaaten besser vertreten zu können.
    • Die Bundesregierung sollte sich für eine stärkere Zusammenarbeit mit anderen europäischen Ländern einsetzen, um gemeinsame Sicherheitsziele zu erreichen.
    • Ein stärkerer Schwerpunkt auf europäische Verteidigungspolitik würde nicht bedeuten, dass Deutschland seine eigenen militärischen Fähigkeiten aufgeben würde, sondern dass es seine Fähigkeiten besser mit anderen europ

...


⚠️  Keine relevanten Informationen gefunden. Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren.



Du:  Wie steht die FDP zur Mietpreisbremse?



⚠️  Keine relevanten Informationen gefunden. Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren.



Du:  Wie steht Christian Lindner zur Mietpreisbremse?



────────────────────────────────────────────────────────────
👤  Christian Lindner (FDP · Bundesvorsitzender · Opposition)
📅  2022-02-15  |  Legislaturperiode: 20 | Sitzung: 42 
🎯  Konfidenz: high
────────────────────────────────────────────────────────────

📌  Claim:
    Die Mietpreisbremse ist nicht notwendig, da sie die Marktwirtschaft stört und zu einer Knappheit an Wohnraum führt.

💬  Argumente:
    • Die Mietpreisbremse würde die Marktwirtschaft stören und zu einer Knappheit an Wohnraum führen.
    • Die Bürgerinitiative gegen die Mietpreisbremse hat bereits 100.000 Unterschriften gesammelt.
    • Die Bundesregierung betont, dass die Mietpreisbremse notwendig ist, um die Wohnraumknappheit zu bekämpfen und die Mieten zu stabilisieren.
    • Die Mietpreisbremse ist ein Beispiel für die übermäßige Regulierung der Wirtschaft durch die Bundesregierung.
────────────────────────────────────────────────────────────



### error log
- don't generate output if query is unclear/ ill-formulated / lacking essential components
- how is the topic introduced for which to retrieve claim/premises ? Query should filter the database preemptively - so query + context in prompt!
- MPs can be MGs, as well (check data!)
- "query" yields expected result, query (w/o "") yields empty result (test: example prompt 1)

### erroneous query log
- Wie steht die FDP zur Mietpreisbremse?
- Wie steht die Regierung zur Mietpreisbremse?
- Wie steht die CDU zum Brexit? 
- Was sagt Angela Merkel zur Covid-19 Situation? 
- Was sind Angela Merkels Argumente für Corona-Hilfen?
- Was sind die Argumente der Grünen für das Klimaschutzgesetz? 
- Finde Merkels Argumente zur Maskenaffäre 
- Wie steht Angela Merkel zum Thema Maskenpflicht? 
- Welche Position vertritt Svenja Schulze zum Klimaschutzgesetz?
- Welche Position vertritt Gregor Gysi zum Atomwaffenverbotsvertrag? (19/207 -> should be retrievable!)


### expected query log
- Welche Position vertritt Svenja Schulze zur CO2-Bepreisung ? (session missing!, correct output, wrong speaker)
- Welche Position vertritt die Bundesregierung zur nuklearen Abrüstung? (session missing !, insufficient output, speaker is MP + MG?)
- "Welche Position vertritt Gregor Gysi zur europäischen Verteidigungspolitik?" (w/o " " different, but correct result)



### nice-to-have
- if no info directly available, give broader context: government instead of Schulze's stance 
    (Welche Position vertritt Svenja Schulze zur CO2-Bepreisung ?)
- topic overview: to increase precision